# 실습 03 · 분류로 '뇌로 가는 약'인지 예측하기
### 혈액뇌장벽(BBB) 투과 여부를 AI 로 판별

**왜 중요한가 (현장 맥락)**
뇌질환 약(치매·우울증 등)은 **혈액뇌장벽(BBB)** 을 통과해야 효과가 있습니다.
반대로 뇌에 작용하면 안 되는 약은 통과하면 부작용이 됩니다.
분자 구조만 보고 **"뇌로 갈 수 있는가(Yes/No)"** 를 예측하는 것은
신약개발 초기의 대표적 **분류(classification)** 문제입니다.

**회귀 vs 분류**
- 회귀(예제 02): 숫자를 예측 (용해도 = -3.2)
- 분류(이 예제): **범주를 예측** (통과 O / 통과 X)


In [ ]:
!pip install rdkit -q
import pandas as pd, numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Draw

# BBBP 데이터: 2000여 개 분자 + 뇌 투과 여부(p_np: 1=통과, 0=미통과)
url = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv"
df = pd.read_csv(url)
print("분자 수:", len(df))
print("통과(1)/미통과(0) 비율:")
print(df["p_np"].value_counts())
df[["name","smiles","p_np"]].head()

## 1. 특성 추출
BBB 투과와 관련 깊은 특성들을 계산합니다.
경험적으로 **작고, 적당히 지용성이며, 극성 표면적(TPSA)이 낮은** 분자가 뇌로 잘 갑니다.


In [ ]:
def featurize(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    return {
        "MolWt"     : Descriptors.MolWt(mol),
        "LogP"      : Crippen.MolLogP(mol),
        "TPSA"      : Descriptors.TPSA(mol),        # 뇌 투과의 핵심 지표
        "HDonors"   : Descriptors.NumHDonors(mol),
        "HAcceptors": Descriptors.NumHAcceptors(mol),
        "RotBonds"  : Descriptors.NumRotatableBonds(mol),
    }

feat = df["smiles"].apply(featurize).apply(pd.Series)
data = pd.concat([feat, df["p_np"]], axis=1).dropna()
X = data.drop(columns="p_np"); y = data["p_np"]
print("특성 표:", X.shape)
X.head()

In [ ]:
# 통과/미통과 그룹의 TPSA 분포 비교 (통과 분자는 TPSA가 낮음을 눈으로 확인)
import matplotlib.pyplot as plt
plt.figure(figsize=(7,4))
plt.hist(data[data.p_np==1]["TPSA"], bins=30, alpha=0.6, label="통과(1)")
plt.hist(data[data.p_np==0]["TPSA"], bins=30, alpha=0.6, label="미통과(0)")
plt.xlabel("TPSA (극성 표면적)"); plt.ylabel("분자 수")
plt.legend(); plt.title("뇌 투과 여부에 따른 TPSA 분포"); plt.show()

## 2. 학습/평가 분리 & 분류 모델 학습
`stratify=y` 로 **통과/미통과 비율을 학습·평가에서 동일하게** 유지합니다 (불균형 데이터 필수 습관).


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# 로지스틱회귀(기본) vs 랜덤포레스트(강력) 비교
models = {
    "로지스틱회귀" : LogisticRegression(max_iter=1000),
    "랜덤포레스트" : RandomForestClassifier(n_estimators=300, random_state=42),
}
for name, m in models.items():
    m.fit(X_train, y_train)
    acc = m.score(X_test, y_test)
    print(f"{name:12s}  정확도(accuracy) = {acc:.3f}")

## 3. ⭐ 분류는 '정확도'만 보면 안 됩니다
데이터가 불균형(통과가 76%)이면 "무조건 통과"라고 찍어도 정확도 76%가 나옵니다.
그래서 현장에서는 **혼동행렬**과 **정밀도/재현율**을 함께 봅니다.

| 지표 | 의미 | 제약 현장 예시 |
|---|---|---|
| 정밀도(Precision) | 통과라 예측한 것 중 진짜 통과 비율 | 헛수고 줄이기 |
| 재현율(Recall) | 진짜 통과 분자를 놓치지 않은 비율 | 좋은 후보 놓치지 않기 |
| ROC-AUC | 전반적 판별력 (1에 가까울수록 좋음) | 모델 비교 표준 지표 |


In [ ]:
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, RocCurveDisplay)
rf = models["랜덤포레스트"]
pred = rf.predict(X_test)
proba = rf.predict_proba(X_test)[:,1]   # 통과일 확률

print("혼동행렬 [행=실제, 열=예측]:")
print(confusion_matrix(y_test, pred))
print("\n상세 리포트:")
print(classification_report(y_test, pred, target_names=["미통과","통과"]))
print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))

In [ ]:
# ROC 곡선 그리기 (곡선이 왼쪽 위로 붙을수록 좋은 모델)
RocCurveDisplay.from_predictions(y_test, proba)
plt.title("BBB 투과 예측 ROC 곡선"); plt.show()

## 4. 새 분자로 예측 + 확률
확률까지 보여주면 "얼마나 확신하는지"를 알 수 있어 현장 의사결정에 유용합니다.


In [ ]:
tests = {
    "카페인(뇌 자극→통과 예상)": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "도파민(통과 어려움)"      : "NCCc1ccc(O)c(O)c1",
    "디아제팜(신경안정제→통과)": "CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc21",
}
rows=[]
for name, smi in tests.items():
    f = pd.DataFrame([featurize(smi)])[X.columns]
    p = rf.predict_proba(f)[0,1]
    rows.append({"분자":name, "통과 확률":f"{p:.0%}", "판정":"통과" if p>0.5 else "미통과"})
pd.DataFrame(rows)

## 정리 & 현장 응용
- 분류 문제는 **정확도만 보지 말고** 혼동행렬·정밀도·재현율·AUC 를 함께 볼 것
- 같은 틀로 **독성 여부, hERG 심장독성, 활성/비활성, 합성가능성** 등 O/X 예측에 재사용
- 확률 출력으로 후보물질 **우선순위**를 매길 수 있음
